# RSNA Knee: сабмит на Kaggle

Делает `submission.csv` из весов, обученных в `02_report_labels_mil.ipynb`. Ничего не учит.

Test готовится ровно так же, как обучающие данные:
1. те же функции, что в `03_build_cache_kaggle.ipynb`, читают DICOM test и пишут «сырые» стеки срезов во временную папку
   (такие же файлы `<ID>.npz`, как в кэше для обучения);
2. тот же класс `KneeStudyDataset`, что в 02, выбирает окна и нормализует (режим проверки: окна фиксированные, без аугментаций);
3. каждая модель предсказывает обычный и зеркальный вид, ранги всех моделей усредняются.

Настройки (`IMG`, `CROP_MM`, `MAX_SLICES`, окна, энкодер) берутся из файлов весов.

## Как запустить

1. Kaggle -> New Notebook -> File -> Import Notebook -> этот файл.
2. Input: соревнование `rsna-knee-abnormality-detection` и датасет(ы) с весами `fold*.pth`
   (можно несколько, например ViT-S и ViT-B: ранги всех моделей усредняются).
3. Желательно подключить датасет с декодерами сжатых DICOM (раздел 1).
4. Accelerator **GPU**, Internet **выключен**. Submit.

In [ ]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "4")

import gc
import re
import math
import time
import glob
import json
import sys
import hashlib
import textwrap
import warnings
from copy import deepcopy
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import cv2
import numpy as np
import pandas as pd
import pydicom
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader

warnings.filterwarnings("ignore")
T0 = time.time()

def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)

try:
    display
except NameError:
    display = print

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log(f"устройство: {DEV}, torch {torch.__version__}, timm {timm.__version__}")

## Настройки и веса

In [ ]:
import fnmatch

INPUT = Path("/kaggle/input")
DATA_DIR = INPUT / "competitions" / "rsna-knee-abnormality-detection"
if not DATA_DIR.exists():
    DATA_DIR = INPUT / "rsna-knee-abnormality-detection"
OUT_DIR = Path("/kaggle/working")

SERIES_THREADS = 24        # сколько серий читаем параллельно
EVAL_BATCH = 4
USE_AMP = DEV.type == "cuda"
AMP_DTYPE = torch.float16
TTA_MIRROR = True          # усредняем обычный и зеркальный вид колена
TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]

# все fold*.pth из подключённых датасетов (в папки соревнования не заходим: там сотни тысяч файлов)
FOLD_FILES = []
for root, dirs, files in os.walk(INPUT, followlinks=True):
    dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
    FOLD_FILES += [Path(root) / f for f in files if fnmatch.fnmatch(f, "fold*.pth")]
FOLD_FILES = sorted(FOLD_FILES)
assert FOLD_FILES, "Нет весов fold*.pth: подключите датасет с весами в Input"

# настройки подготовки данных и модели берём из первого файла весов
CFG = torch.load(FOLD_FILES[0], map_location="cpu", weights_only=False)["config"]
IMG, CROP_MM, MAX_SLICES = CFG["IMG"], CFG["CROP_MM"], CFG["MAX_SLICES"]
K_WIN, WIN_BAND, FILL_MISSING_SLOTS = CFG["K_WIN"], tuple(CFG["WIN_BAND"]), CFG["FILL_MISSING_SLOTS"]
BACKBONE, UNFREEZE_LAST = CFG["BACKBONE"], CFG["UNFREEZE_LAST"]
HIDDEN, TF_DEPTH = CFG["HIDDEN"], CFG["TF_DEPTH"]
WEIGHTS_PATH = False       # предобученные веса не нужны: всё берётся из fold*.pth
for fp in FOLD_FILES:
    log(f"веса: {fp}")
log(f"настройки из весов: IMG={IMG}, CROP_MM={CROP_MM}, MAX_SLICES={MAX_SLICES}, K_WIN={K_WIN}, BACKBONE={BACKBONE}")

## 1. Декодеры сжатых DICOM

Часть DICOM-файлов хранится сжатой (JPEG, JPEG-2000, JPEG-LS), и сам `pydicom` их не распакует.
По словам автора ноутбука 0.958, в скрытом test такие файлы есть. Если среди подключённых датасетов есть
`.whl`-файлы декодеров, ячейка ставит их без интернета и проверяет 100 файлов test.

In [ ]:
import importlib, subprocess

wheels = [w for pat in ("/kaggle/input/*/*.whl", "/kaggle/input/*/*/*.whl", "/kaggle/input/*/*/*/*.whl")
          for w in glob.glob(pat) if re.search(r"pylibjpeg|gdcm|jpegls", os.path.basename(w), re.I)]
if wheels:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps"] + wheels, check=False)
    importlib.invalidate_caches()
log(f"колёс декодеров найдено: {len(wheels)}")

test_series = pd.read_csv(DATA_DIR / "test_series.csv")
n_fail = n_comp = 0
for r in test_series.head(100).itertuples():
    d = DATA_DIR / "test_series" / str(r.StudyInstanceUID) / str(r.SeriesInstanceUID)
    fs = [f for f in os.listdir(d) if f.endswith(".dcm")] if d.is_dir() else []
    if not fs:
        continue
    try:
        ds = pydicom.dcmread(d / fs[0])
        n_comp += ds.file_meta.TransferSyntaxUID.is_compressed
        ds.pixel_array
    except Exception:
        n_fail += 1
log(f"проверка test: сжатых {n_comp}, не прочитались {n_fail}")

## 2. Dataset

Тот же класс, что в `02_report_labels_mil.ipynb`. Записывается в файл и импортируется.

In [ ]:
%%writefile knee_dataset.py
"""Датасет RSNA Knee: одно исследование -> окна из 3 соседних срезов во всех сериях.

Этот файл пишет ячейка %%writefile в ноутбуке. Отдельный файл нужен потому, что DataLoader на macOS
запускает воркеры как новые процессы Python, и они могут взять класс только из файла, а не из ячейки.
"""
import math
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

# Шесть слотов: (имя, плоскость, Fat_Suppression). Номер слота = номер в списке.
SLOTS = [
    ("SAG_FS", "Sagittal", 1), ("COR_FS", "Coronal", 1), ("AX_FS", "Axial", 1),
    ("SAG_NOFS", "Sagittal", 0), ("COR_NOFS", "Coronal", 0), ("AX_NOFS", "Axial", 0),
]
N_SLOT = len(SLOTS)
SAG_SLOTS = [k for k, (_, plane, _) in enumerate(SLOTS) if plane == "Sagittal"]
FILL_PAIRS = ((0, 3), (1, 4), (2, 5))      # одна плоскость: с подавлением жира и без
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
DEFAULT_AUG = dict(rot_deg=10.0, scale=0.10, shift=0.06, gamma=0.25, gain=0.10)


def window_centres(n, k, band, rng=None):
    """Центры k окон в стеке из n срезов, в полосе band (доли стека).

    rng=None (проверка): окна стоят равномерно, всегда одинаково.
    rng задан (обучение): полоса делится на k кусков, в каждом центр выбирается случайно,
    поэтому каждую эпоху модель видит немного другие срезы (аугментация вдоль стека).
    """
    a = max(1.0, band[0] * (n - 1))          # c-1 и c+1 должны существовать
    b = min(n - 2.0, band[1] * (n - 1))
    if b < a:
        a = b = (n - 1) / 2
    if rng is None:
        c = np.linspace(a, b, k)
    else:
        edges = np.linspace(a, b, k + 1)
        c = edges[:-1] + rng.random(k) * (edges[1:] - edges[:-1])
    return np.clip(np.round(c).astype(int), 1, max(1, n - 2))


def augment(x, rot_deg, scale, shift, gamma, gain, gen):
    """Случайные искажения, не меняющие диагноз: поворот, сдвиг, зум внутрь, контраст, яркость.

    x: float [N, 3, H, W] в [0, 1]. Вертикального флипа нет: бедро и голень поменялись бы местами.
    """
    n = x.shape[0]
    r = lambda *s: torch.rand(*s, generator=gen)
    rot = (r(n) - 0.5) * 2 * math.radians(rot_deg)
    sc = 1.0 + r(n) * scale                     # > 1 -> зум внутрь (края не дорисовываются)
    tx, ty = (r(n) - 0.5) * 2 * shift, (r(n) - 0.5) * 2 * shift
    cos, sin = torch.cos(rot) / sc, torch.sin(rot) / sc
    theta = torch.zeros(n, 2, 3)
    theta[:, 0, 0], theta[:, 0, 1], theta[:, 0, 2] = cos, -sin, tx
    theta[:, 1, 0], theta[:, 1, 1], theta[:, 1, 2] = sin, cos, ty
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    g = torch.exp((r(n, 1, 1, 1) - 0.5) * 2 * gamma)        # контраст (гамма)
    k = 1.0 + (r(n, 1, 1, 1) - 0.5) * 2 * gain              # яркость
    return (x.clamp_min(1e-4) ** g * k).clamp(0, 1)


def mirror_batch(x, pos):
    """Зеркало колена для готового батча (TTA на проверке).

    x [B, S, K, 3, H, W], pos [B, S, K]. Сагиттальные: обратный порядок окон и срезов внутри окна;
    корональные и аксиальные: отражение по горизонтали.
    """
    x, pos = x.clone(), pos.clone()
    for s in range(x.shape[1]):
        if s in SAG_SLOTS:
            x[:, s] = x[:, s].flip(1).flip(2)
            pos[:, s] = 1 - pos[:, s].flip(-1)
        else:
            x[:, s] = x[:, s].flip(-1)
    return x, pos


class KneeStudyDataset(torch.utils.data.Dataset):
    """Один пример = одно исследование.

    study_ids: ID исследований.
    source: папка с файлами <ID>.npz ИЛИ словарь {ID: {"slot0": [n, H, W] uint8, ...}} (так делает сабмит).
    labels, weights: [N, 12] (мягкие метки и веса) или None, если меток нет (test).
    train: True -> случайные окна, зеркало и аугментации; False -> фиксированные окна, без искажений.
    k_windows: сколько окон по 3 соседних среза брать из каждой серии.

    Возвращает x [6, K, 3, H, W] (нормализовано как ImageNet), mask [6], pos [6, K], y [12], w [12].
    """

    def __init__(self, study_ids, source, labels=None, weights=None, train=False, k_windows=4,
                 win_band=(0.08, 0.92), fill_missing=True, mirror_p=0.5, aug=None):
        self.ids = [str(s) for s in study_ids]
        self.source = source
        self.labels = None if labels is None else np.asarray(labels, np.float32)
        self.weights = None if weights is None else np.asarray(weights, np.float32)
        self.train, self.k = train, k_windows
        self.win_band, self.fill_missing, self.mirror_p = tuple(win_band), fill_missing, mirror_p
        self.aug = DEFAULT_AUG if aug is None else aug

    def __len__(self):
        return len(self.ids)

    def load_stacks(self, sid):
        # -> {номер слота: uint8 [n_срезов, H, W]}: только найденные серии
        if isinstance(self.source, dict):
            raw = self.source[sid]
        else:
            with np.load(Path(self.source) / f"{sid}.npz") as z:
                raw = {key: z[key] for key in z.files}
        return {int(key[4:]): arr for key, arr in raw.items() if len(arr) > 0}

    def __getitem__(self, i):
        sid = self.ids[i]
        stacks = self.load_stacks(sid)
        h, w = next(iter(stacks.values())).shape[1:] if stacks else (224, 224)
        # отдельный генератор на каждый пример: случайность не повторяется между воркерами
        seed = int(torch.randint(0, 2 ** 31 - 1, (1,)).item())
        rng, gen = np.random.default_rng(seed), torch.Generator().manual_seed(seed)

        # 1. Нет серии с fat-sat (или без) -> подставляем её пару из той же плоскости
        if self.fill_missing:
            for a, b in FILL_PAIRS:
                if a not in stacks and b in stacks:
                    stacks[a] = stacks[b]
                elif b not in stacks and a in stacks:
                    stacks[b] = stacks[a]

        # 2. Зеркало колена (только обучение): левое колено превращается в «правое».
        #    Сагиттальная серия идёт поперёк колена -> разворачиваем порядок срезов;
        #    корональная и аксиальная -> отражаем каждый срез по горизонтали.
        if self.train and rng.random() < self.mirror_p:
            stacks = {k: (s[::-1] if k in SAG_SLOTS else s[:, :, ::-1]) for k, s in stacks.items()}

        # 3. Окна: k центров в полосе стека, окно = срезы [c-1, c, c+1] -> 3 канала картинки
        x = np.zeros((N_SLOT, self.k, 3, h, w), np.uint8)
        mask = np.zeros(N_SLOT, np.float32)
        pos = np.zeros((N_SLOT, self.k), np.float32)
        for k, s in stacks.items():
            n = len(s)
            c = window_centres(n, self.k, self.win_band, rng if self.train else None)
            x[k] = s[np.stack([c - 1, c, c + 1], 1)]       # [k окон, 3 среза, H, W]
            mask[k] = 1.0
            pos[k] = c / max(n - 1, 1)                     # где окно в стеке: 0 = начало, 1 = конец

        # 4. В float [0, 1], аугментации (только обучение), нормализация ImageNet
        t = torch.from_numpy(x).float().div_(255.0).view(N_SLOT * self.k, 3, h, w)
        if self.train:
            t = augment(t, gen=gen, **self.aug)
        t = ((t - IMAGENET_MEAN) / IMAGENET_STD).view(N_SLOT, self.k, 3, h, w)

        y = self.labels[i] if self.labels is not None else np.zeros(12, np.float32)
        wt = self.weights[i] if self.weights is not None else np.ones(12, np.float32)
        return t, torch.from_numpy(mask), torch.from_numpy(pos), torch.from_numpy(y), torch.from_numpy(wt)

In [ ]:
sys.path.insert(0, str(Path.cwd()))
from knee_dataset import KneeStudyDataset, mirror_batch, N_SLOT, SAG_SLOTS

## 3. Чтение test: те же функции, что в `03_build_cache_kaggle.ipynb`

Для каждого исследования test: выбор серий -> все срезы по порядку -> ориентация -> кроп 140 мм -> 224 px -> uint8.
Каждое исследование записывается во временную папку как `<ID>.npz`, ровно как в кэше для обучения. Временная папка
не попадает в Output, а память не растёт с размером скрытого test.

In [ ]:
SLOTS = [  # (имя, плоскость, Fat_Suppression)
    ("SAG_FS", "Sagittal", 1), ("COR_FS", "Coronal", 1), ("AX_FS", "Axial", 1),
    ("SAG_NOFS", "Sagittal", 0), ("COR_NOFS", "Coronal", 0), ("AX_NOFS", "Axial", 0),
]


def count_files(d):
    try:
        return sum(1 for e in os.scandir(d) if e.name.endswith(".dcm"))
    except OSError:
        return 0


def pick_series(series_df):
    # -> таблица «исследование, слот -> серия», одна строка на найденную комбинацию
    df = series_df.copy()
    df["dir"] = [str(DATA_DIR / "test_series" / str(a) / str(b))
                 for a, b in zip(df["StudyInstanceUID"], df["SeriesInstanceUID"])]
    with ThreadPoolExecutor(SERIES_THREADS) as pool:
        df["n_files"] = list(pool.map(count_files, df["dir"]))
    rows = []
    for study, g in df.groupby("StudyInstanceUID"):
        for k, (name, plane, fs) in enumerate(SLOTS):
            cand = g[(g["Anatomical_Plane"] == plane) & (g["Fat_Suppression"].astype(int) == fs)
                     & (g["n_files"] >= 3)]
            if len(cand):
                r = cand.sort_values("n_files", ascending=False).iloc[0]
                rows.append({"StudyInstanceUID": study, "slot": k, "slot_name": name,
                             "SeriesInstanceUID": r["SeriesInstanceUID"], "dir": r["dir"],
                             "plane": plane, "n_files": int(r["n_files"])})
    return pd.DataFrame(rows)

In [ ]:
HDR_TAGS = ["ImagePositionPatient", "ImageOrientationPatient", "PixelSpacing", "InstanceNumber"]
PLANES = ["Sagittal", "Coronal", "Axial"]
# Каноническая ориентация (из ноутбука 0.958). Для плоскости, у которой нормаль идёт вдоль оси
# 0 (x, лево-право) / 1 (y, перёд-зад) / 2 (z, верх-низ), задаём:
# (знак порядка срезов, (ось вдоль столбцов картинки, знак), (ось вдоль строк, знак))
# Система координат DICOM: x -> к ЛЕВОМУ боку пациента, y -> к спине, z -> к голове.
CANON = {
    0: (+1, (1, +1), (2, -1)),   # сагиттальная: слева направо по картинке = спереди назад, вниз = от головы
    1: (+1, (0, +1), (2, -1)),   # корональная: по картинке справа пациента налево, вниз = от головы
    2: (-1, (0, +1), (1, +1)),   # аксиальная: срезы сверху вниз, по картинке вниз = спереди назад
}


def _vec(v, n):
    try:
        a = np.asarray([float(x) for x in v], float)
        return a if len(a) >= n else None
    except Exception:
        return None


def read_headers(d):
    recs = []
    for f in sorted(os.listdir(d)):
        if not f.endswith(".dcm"):
            continue
        rec = {"file": f, "ipp": None, "iop": None, "ps": None, "inst": None}
        try:
            h = pydicom.dcmread(os.path.join(d, f), stop_before_pixels=True, force=True,
                                specific_tags=HDR_TAGS)
            rec["ipp"] = _vec(getattr(h, "ImagePositionPatient", []), 3)
            rec["iop"] = _vec(getattr(h, "ImageOrientationPatient", []), 6)
            rec["ps"] = _vec(getattr(h, "PixelSpacing", []), 2)
            rec["inst"] = float(getattr(h, "InstanceNumber", None) or 0)
        except Exception:
            pass
        recs.append(rec)
    return recs


def sort_slices(recs, plane_name):
    # -> (отсортированные записи, номер оси плоскости 0/1/2, IOP для ориентации)
    iop = next((r["iop"] for r in recs if r["iop"] is not None), None)
    if iop is not None:
        normal = np.cross(iop[:3], iop[3:])
        ax = int(np.abs(normal).argmax())
        if all(r["ipp"] is not None for r in recs):
            sgn = CANON[ax][0] * np.sign(normal[ax])
            recs = sorted(recs, key=lambda r: float(np.dot(r["ipp"], normal)) * sgn)
            return recs, ax, iop
    else:
        ax = PLANES.index(plane_name) if plane_name in PLANES else 0
    return sorted(recs, key=lambda r: (r["inst"] or 0, r["file"])), ax, iop


def canon_orient(img, iop, ps, ax):
    # поворачиваем/отражаем срез к канонической ориентации; ps (размер пикселя) меняем вместе с ним
    if iop is None:
        return img, ps
    r, c = np.array(iop[:3]), np.array(iop[3:])
    _, (cax, csgn), (rax, rsgn) = CANON[ax]
    if abs(r[cax]) < abs(r[rax]):           # строки и столбцы перепутаны -> транспонируем
        img, r, c, ps = img.T, c, r, ps[::-1]
    if r[cax] * csgn < 0:
        img = img[:, ::-1]                  # отражаем по горизонтали
    if c[rax] * rsgn < 0:
        img = img[::-1]                     # отражаем по вертикали
    return np.ascontiguousarray(img), ps


def crop_resize(img, ps):
    # квадрат CROP_MM мм вокруг центра -> IMG×IMG; если снимок меньше, дополняем нулями
    out = np.zeros((IMG, IMG), np.float32)
    mm_per_px = CROP_MM / IMG
    h, w = img.shape
    ch = min(h, int(round(CROP_MM / ps[0])))
    cw = min(w, int(round(CROP_MM / ps[1])))
    y0, x0 = (h - ch) // 2, (w - cw) // 2
    crop = np.ascontiguousarray(img[y0:y0 + ch, x0:x0 + cw])
    nh = max(1, min(IMG, int(round(ch * ps[0] / mm_per_px))))
    nw = max(1, min(IMG, int(round(cw * ps[1] / mm_per_px))))
    interp = cv2.INTER_AREA if nh < ch else cv2.INTER_LINEAR   # AREA - без «лесенки» при уменьшении
    res = cv2.resize(crop, (nw, nh), interpolation=interp)
    oy, ox = (IMG - nh) // 2, (IMG - nw) // 2
    out[oy:oy + nh, ox:ox + nw] = res
    return out


def decode(path):
    ds = pydicom.dcmread(path, force=True)
    a = ds.pixel_array.astype(np.float32)
    if a.ndim == 3:                         # редкий «мультикадровый» файл - берём средний кадр
        a = a[len(a) // 2]
    a = a * float(getattr(ds, "RescaleSlope", 1) or 1) + float(getattr(ds, "RescaleIntercept", 0) or 0)
    if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
        a = a.max() - a                     # в MONOCHROME1 «белое» и «чёрное» перевёрнуты
    return a


def read_stack(d, plane_name):
    # серия -> uint8 [n_срезов, IMG, IMG] по порядку в теле, или None, если ничего не прочиталось
    recs = read_headers(d)
    if not recs:
        return None
    recs, ax, iop = sort_slices(recs, plane_name)
    n = len(recs)
    idx = np.unique(np.linspace(0, n - 1, min(n, MAX_SLICES)).round().astype(int))

    imgs = {}
    for i in idx:
        try:
            imgs[int(i)] = decode(os.path.join(d, recs[int(i)]["file"]))
        except Exception:
            imgs[int(i)] = None
    ok = [i for i, a in imgs.items() if a is not None]
    if not ok:
        return None
    for i in imgs:                          # сломанный срез -> ближайший целый
        if imgs[i] is None:
            imgs[i] = imgs[min(ok, key=lambda j: abs(j - i))]

    ps_default = next((r["ps"] for r in recs if r["ps"] is not None), None)
    planes = []
    for i in idx:
        a = imgs[int(i)]
        ps = recs[int(i)]["ps"] if recs[int(i)]["ps"] is not None else ps_default
        if ps is None:                      # нет размера пикселя -> считаем, что снимок = кропу
            ps = np.array([CROP_MM / max(a.shape)] * 2)
        planes.append(canon_orient(a, iop, list(ps), ax))

    allpx = np.concatenate([a[::4, ::4].ravel() for a, _ in planes])
    lo, hi = np.percentile(allpx, [0.5, 99.5])
    stack = np.stack([crop_resize(np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1), ps) for a, ps in planes])
    return (stack * 255).round().astype(np.uint8)

In [ ]:
import tempfile

TEST_DIR = Path(tempfile.gettempdir()) / "test_studies"   # временная папка, не Output
TEST_DIR.mkdir(parents=True, exist_ok=True)
test_df = pd.read_csv(DATA_DIR / "test.csv")
TEST_INDEX = pick_series(test_series)
TEST_INDEX["StudyInstanceUID"] = TEST_INDEX["StudyInstanceUID"].astype(str)


def build_study(item):
    # одно исследование test -> TEST_DIR/<ID>.npz {"slot0": [n, IMG, IMG], ...}; None, если ничего не прочиталось
    sid, g = item
    stacks = {f"slot{r.slot}": read_stack(r.dir, r.plane) for r in g.itertuples(index=False)}
    stacks = {k: v for k, v in stacks.items() if v is not None}
    if not stacks:
        return None
    np.savez(TEST_DIR / f"{sid}.npz", **stacks)
    return sid


with ThreadPoolExecutor(SERIES_THREADS) as pool:
    done = set(pool.map(build_study, TEST_INDEX.groupby("StudyInstanceUID"))) - {None}
TEST_IDS = [s for s in test_df["StudyInstanceUID"].astype(str) if s in done]
log(f"test: {len(test_df)} исследований, со снимками {len(TEST_IDS)}")

## 4. Модель и предсказание

In [ ]:
def load_backbone(name, weights_path, img):
    # weights_path=False -> пустая архитектура (на инференсе веса придут из чекпойнта фолда)
    kw = dict(num_classes=0)                # num_classes=0 -> сеть отдаёт вектор признаков, без головы
    if name.startswith(("vit_", "eva", "beit")):
        kw["img_size"] = img                # у ViT позиционные эмбеддинги зависят от размера входа
    if weights_path is False:
        return timm.create_model(name, pretrained=False, **kw)
    if weights_path and os.path.exists(weights_path):
        m = timm.create_model(name, pretrained=False, **kw)
        sd = torch.load(weights_path, map_location="cpu", weights_only=False)
        for key in ("state_dict", "model"):               # бывают обёртки {"model": {...}}
            if isinstance(sd, dict) and key in sd and isinstance(sd[key], dict):
                sd = sd[key]
        sd = {k.removeprefix("module.").removeprefix("model."): v for k, v in sd.items()}
        sd = {k: v for k, v in sd.items() if not k.startswith(("head.fc.", "fc.", "classifier."))
              and not (k.startswith("head.") and name.startswith("vit_"))}
        if "pos_embed" in sd and hasattr(m, "pos_embed") and sd["pos_embed"].shape != m.pos_embed.shape:
            # веса DINOv2 сохранены под 518px; под наш IMG интерполируем позиционные эмбеддинги
            from timm.layers import resample_abs_pos_embed
            sd["pos_embed"] = resample_abs_pos_embed(sd["pos_embed"], new_size=m.patch_embed.grid_size,
                                                     num_prefix_tokens=m.num_prefix_tokens)
        missing, unexpected = m.load_state_dict(sd, strict=False)
        assert not missing and not unexpected, (
            f"веса не подходят к {name}: missing={missing[:5]} unexpected={unexpected[:5]}")
        log(f"энкодер {name}: веса из файла, все ключи совпали")
        return m
    log(f"энкодер {name}: файла весов нет - скачиваю через timm (нужен Internet)")
    return timm.create_model(name, pretrained=True, **kw)


def set_trainable(bb, n_last):
    # всё заморозить, потом открыть «последнюю часть» сети
    for p in bb.parameters():
        p.requires_grad = False
    if hasattr(bb, "blocks"):               # ViT: последние n_last блоков + финальная нормализация
        parts = list(bb.blocks)[-n_last:] + [getattr(bb, "norm", None)]
    elif hasattr(bb, "stages"):             # ConvNeXt и похожие CNN: две последние стадии + голова
        parts = list(bb.stages)[-2:] + [getattr(bb, "head", None), getattr(bb, "norm_pre", None)]
    else:
        parts = [bb]
    for part in parts:
        if part is not None:
            for p in part.parameters():
                p.requires_grad = True


class StudyMIL(nn.Module):
    def __init__(self, backbone=BACKBONE, weights=WEIGHTS_PATH, unfreeze_last=UNFREEZE_LAST,
                 hidden=HIDDEN, tf_depth=TF_DEPTH):
        super().__init__()
        self.backbone = load_backbone(backbone, weights, IMG)
        set_trainable(self.backbone, unfreeze_last)
        d = self.backbone.num_features
        self.proj = nn.Sequential(nn.Dropout(0.1), nn.Linear(d, hidden), nn.LayerNorm(hidden))
        self.slot_emb = nn.Embedding(N_SLOT, hidden)            # «это сагиттальная FS» и т.п.
        self.pos_emb = nn.Sequential(nn.Linear(1, hidden), nn.GELU(), nn.Linear(hidden, hidden))
        if tf_depth:
            layer = nn.TransformerEncoderLayer(hidden, 4, hidden * 2, dropout=0.1,
                                               batch_first=True, norm_first=True)
            self.tf = nn.TransformerEncoder(layer, tf_depth, enable_nested_tensor=False)
        else:
            self.tf = None
        # внимание: для каждого окна 12 чисел «насколько это окно важно для патологии c»
        self.att = nn.Sequential(nn.Linear(hidden, 128), nn.Tanh(), nn.Dropout(0.2),
                                 nn.Linear(128, len(TARGETS)))
        self.cls_w = nn.Parameter(torch.randn(len(TARGETS), hidden) * 0.02)   # свой выход у каждой патологии
        self.cls_b = nn.Parameter(torch.zeros(len(TARGETS)))

    def forward(self, x, mask, pos):
        # x: float [B, S, K, 3, H, W] (уже нормализован в Dataset)   mask: [B, S]   pos: [B, S, K]   ->   логиты [B, 12]
        B, S, K = x.shape[:3]
        keep = mask.reshape(-1) > 0.5                          # какие (исследование, слот) есть
        feat = None
        if keep.any():
            xs = x.reshape(B * S, K, *x.shape[3:])[keep].flatten(0, 1)   # только существующие картинки
            f = self.backbone(xs)                              # [n_картинок, d]
            feat = f.new_zeros(B * S, K, f.shape[-1])
            feat[keep] = f.view(-1, K, f.shape[-1])
        else:
            feat = x.new_zeros((B * S, K, self.backbone.num_features))
        h = self.proj(feat.view(B, S, K, -1))
        h = h + self.slot_emb(torch.arange(S, device=x.device)).view(1, S, 1, -1)
        h = h + self.pos_emb(pos.unsqueeze(-1).to(h.dtype))
        h = h.view(B, S * K, -1)                               # все окна исследования в одну «строку»
        pad = ~(mask > 0.5).unsqueeze(-1).expand(B, S, K).reshape(B, S * K)   # True = окна нет
        pad = pad & ~pad.all(1, keepdim=True)                  # исследование без серий: не маскируем всё
        if self.tf is not None:
            h = self.tf(h, src_key_padding_mask=pad)
        a = self.att(h).float().masked_fill(pad.unsqueeze(-1), -1e4).softmax(1)   # [B, окна, 12]
        pooled = torch.einsum("btc,btd->bcd", a.to(h.dtype), h)                   # [B, 12, hidden]
        return (pooled * self.cls_w).sum(-1) + self.cls_b

In [ ]:
@torch.no_grad()
def predict(model, loader, tta_mirror=TTA_MIRROR):
    # -> вероятности [число исследований в loader, 12] в порядке loader (shuffle=False)
    model.eval()
    out = []
    for x, m, p, _, _ in loader:
        x, m, p = x.to(DEV), m.to(DEV), p.to(DEV)
        with torch.autocast(DEV.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            prob = torch.sigmoid(model(x, m, p).float())
            if tta_mirror:                                 # второй взгляд: зеркальное колено
                xm, pm = mirror_batch(x, p)
                prob = 0.5 * (prob + torch.sigmoid(model(xm, m, pm).float()))
        out.append(prob.cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, 12), np.float32)

## 5. Сабмит

1. Сразу пишем файл со всеми 0.5: если дальше что-то упадёт, на диске всё равно будет валидный сабмит.
2. Каждая модель предсказывает test (обычный и зеркальный вид), предсказания переводятся в ранги и усредняются.
3. Склеиваем с `test.csv` по `StudyInstanceUID`. Исследование без снимков получает 0.5.

In [ ]:
bench = test_df[["StudyInstanceUID"]].copy()
for t in TARGETS:
    bench[t] = 0.5
bench.to_csv(OUT_DIR / "submission.csv", index=False)

loader = DataLoader(KneeStudyDataset(TEST_IDS, TEST_DIR, train=False, k_windows=K_WIN, win_band=WIN_BAND,
                                     fill_missing=FILL_MISSING_SLOTS),
                    batch_size=EVAL_BATCH, shuffle=False, num_workers=0)
ranks = []
for fp in FOLD_FILES:
    ck = torch.load(fp, map_location="cpu", weights_only=False)
    cfg = ck["config"]
    assert (cfg["IMG"], cfg["CROP_MM"], cfg["K_WIN"]) == (IMG, CROP_MM, K_WIN), f"{fp.name} обучен с другими настройками: {cfg}"
    model = StudyMIL(backbone=cfg["BACKBONE"], weights=False, unfreeze_last=cfg["UNFREEZE_LAST"],
                     hidden=cfg["HIDDEN"], tf_depth=cfg["TF_DEPTH"]).to(DEV)
    model.load_state_dict(ck["model"])
    pr = predict(model, loader)
    ranks.append(pd.DataFrame(pr).rank(pct=True).values)
    log(f"{fp.name}: предсказано {len(TEST_IDS)} исследований")
    del model, ck
    gc.collect()

sub = pd.DataFrame(np.mean(ranks, axis=0), columns=TARGETS)
sub.insert(0, "StudyInstanceUID", TEST_IDS)
sub = test_df[["StudyInstanceUID"]].astype(str).merge(sub, on="StudyInstanceUID", how="left")
sub[TARGETS] = sub[TARGETS].fillna(0.5)
sub.to_csv(OUT_DIR / "submission.csv", index=False)
log(f"submission.csv: {sub.shape}, пустых значений: {int(sub[TARGETS].isna().sum().sum())}")
display(sub.head())